# NB17: Sensitivity Analysis and Visualization

Robustness checks and publication-quality figures for the main findings:
- Primary: MicrobeAtlas soil PGLS (metal KO density → niche breadth, n=603)
- NGSA-informed: AusMicrobiome PGLS (soil Cu/Zn/Pb/Ni → niche breadth, n=482)
- Spatial: Moran's I by biome, mining proximity signal, latitude gradient

**Sensitivity analyses:**
1. FDR correction (BH) for multiple NGSA metal tests
2. NGSA distance threshold sensitivity (30–200 km) via OTU-level re-aggregation
3. Genus detection frequency threshold sensitivity (min detections)

**Figures produced:**
- Fig 1: Primary PGLS scatter (soil genera, n=603)
- Fig 2: AusMicrobiome NGSA forest plot (β ± 95% CI per predictor)
- Fig 3: Genus-level NGSA Cu scatter (raw Cu_ppm vs Levins' B)
- Fig 4: Moran's I by biome (bar chart)
- Fig 5: Mining proximity (binned scatter, Soil MAGs)
- Fig 6: NGSA distance threshold sensitivity (β stability across thresholds)

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
import subprocess
import tempfile
import os
import gzip

warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
BASE    = '/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology'
TREE    = f'{BASE}/data/gtdb_bac_genus_pruned.tree'
PGLS_R  = f'{BASE}/scripts/pgls_mgnify_validation.R'
FIGDIR  = f'{BASE}/data/figures'
RSCRIPT = '/home/hmacgregor/.local/envs/bio_env/bin/Rscript'
os.makedirs(FIGDIR, exist_ok=True)

# ── matplotlib style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'sans-serif',
    'font.size':          11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     0.8,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'figure.dpi':         150,
})

# colour palette
C = dict(
    primary='#2166ac',
    mgnify='#1a9641',
    aus='#d73027',
    null='#aaaaaa',
    soil='#d97c2d',
    marine='#1a75b8',
    sediment='#8b6b4a',
    cu='#c0392b', zn='#e67e22', pb='#8e44ad', ni='#2980b9',
)

print('Setup complete. Figures → ', FIGDIR)
print('Rscript: ', RSCRIPT)

Setup complete. Figures →  /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/figures
Rscript:  /home/hmacgregor/.local/envs/bio_env/bin/Rscript


## 1  FDR correction for NGSA PGLS

In [2]:
# Load NGSA PGLS results
ngsa_res = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_ngsa_pgls_results.csv')
ngsa_res = ngsa_res.sort_values('p_value').reset_index(drop=True)
m = len(ngsa_res)

# Benjamini-Hochberg correction
# Reject H0 for all k such that p_k ≤ (k/m) × α
ngsa_res['rank'] = np.arange(1, m+1)
ngsa_res['bh_threshold_05'] = (ngsa_res['rank'] / m) * 0.05
ngsa_res['bh_threshold_10'] = (ngsa_res['rank'] / m) * 0.10
ngsa_res['q_BH'] = (ngsa_res['p_value'] * m / ngsa_res['rank']).clip(upper=1.0)
# monotone adjustment (BH q-values must be non-decreasing from highest rank)
ngsa_res['q_BH'] = ngsa_res['q_BH'][::-1].cummin()[::-1]

# Find largest k where p_k ≤ BH threshold
sig_05 = ngsa_res[ngsa_res['p_value'] <= ngsa_res['bh_threshold_05']]
sig_10 = ngsa_res[ngsa_res['p_value'] <= ngsa_res['bh_threshold_10']]
print(f'BH correction (m={m} tests):')
print(f'  Significant at FDR=0.05: {len(sig_05)} predictors')
print(f'  Significant at FDR=0.10: {len(sig_10)} predictors → {sig_10["predictor"].tolist()}')
print()

label_map = {
    'ko_per_mb_total_z': 'Metal KO density',
    'ngsa_Cu_ppm_z': 'NGSA Copper',
    'ngsa_Zn_ppm_z': 'NGSA Zinc',
    'ngsa_Pb_ppm_z': 'NGSA Lead',
    'ngsa_Ni_ppm_z': 'NGSA Nickel',
    'ngsa_Co_ppm_z': 'NGSA Cobalt',
}
ngsa_res['label'] = ngsa_res['predictor'].map(label_map)

display_cols = ['label', 'n_taxa', 'beta', 'SE', 'p_value', 'q_BH']
print(ngsa_res[display_cols].to_string(index=False, float_format='%.4f'))

BH correction (m=6 tests):
  Significant at FDR=0.05: 0 predictors
  Significant at FDR=0.10: 3 predictors → ['ngsa_Zn_ppm_z', 'ngsa_Pb_ppm_z', 'ngsa_Ni_ppm_z']

           label  n_taxa    beta     SE  p_value   q_BH
     NGSA Copper     482 -0.0101 0.0043   0.0194 0.0691
       NGSA Zinc     482 -0.0096 0.0043   0.0268 0.0691
       NGSA Lead     482 -0.0089 0.0043   0.0395 0.0691
     NGSA Nickel     482 -0.0087 0.0044   0.0461 0.0691
     NGSA Cobalt     482  0.0019 0.0043   0.6581 0.6674
Metal KO density     482 -0.0023 0.0054   0.6674 0.6674


## 2  Figure 1 — Primary PGLS scatter (MicrobeAtlas soil genera)

In [3]:
# Primary soil PGLS: biome_H_std ~ ko_per_mb_total_z  (n=603, β=−0.023, p<0.001)
soil_df = pd.read_csv(f'{BASE}/data/pgls_input_soil_primary.csv').dropna(
    subset=['biome_H_std', 'ko_per_mb_total_z'])

soil_result = pd.read_csv(f'{BASE}/data/pgls_soil_primary_result.csv')
row = soil_result[soil_result['predictor'] == 'ko_per_mb_total_z'].iloc[0]
beta, se, pval, n_taxa = row['beta'], row['SE'], row['p_value'], int(row['n_taxa'])

# OLS intercept approximation (PGLS regression line anchored at means)
x_mean = soil_df['ko_per_mb_total_z'].mean()
y_mean = soil_df['biome_H_std'].mean()
intercept = y_mean - beta * x_mean

x_range = np.linspace(soil_df['ko_per_mb_total_z'].quantile(0.01),
                       soil_df['ko_per_mb_total_z'].quantile(0.99), 200)
y_fit = intercept + beta * x_range

pval_str = f'p = {pval:.4f}' if pval >= 0.0001 else f'p < 0.0001'

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(soil_df['ko_per_mb_total_z'], soil_df['biome_H_std'],
           alpha=0.35, s=18, color=C['primary'], zorder=2, linewidths=0)
ax.plot(x_range, y_fit, color='#d73027', linewidth=2.2, zorder=3,
        label=f'PGLS line: β = {beta:.3f} (±{1.96*se:.3f})')
ax.axhline(0, color='#cccccc', linewidth=0.6, linestyle='--')
ax.axvline(0, color='#cccccc', linewidth=0.6, linestyle='--')

ax.set_xlabel('Metal resistance KO density per Mb (z-score)', fontsize=12)
ax.set_ylabel('Niche breadth, Levins\' B (standardized)', fontsize=12)
ax.set_title(
    f'MicrobeAtlas soil genera: n = {n_taxa}\n'
    f'β = {beta:.4f}, SE = {se:.4f}, {pval_str}',
    fontsize=11, pad=8)
ax.legend(fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig1_primary_pgls_scatter.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig1_primary_pgls_scatter.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 1 saved.')

Fig 1 saved.


## 3  Figure 2 — AusMicrobiome NGSA forest plot

In [4]:
# Forest plot: β ± 95% CI for each predictor (NGSA metals vs KO/Mb null)
ngsa_res2 = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_ngsa_pgls_results.csv')

# Re-order: null first, then significant metals, then null metal
order = ['ko_per_mb_total_z', 'ngsa_Cu_ppm_z', 'ngsa_Zn_ppm_z',
         'ngsa_Pb_ppm_z', 'ngsa_Ni_ppm_z', 'ngsa_Co_ppm_z']
ngsa_res2 = ngsa_res2.set_index('predictor').loc[order].reset_index()
ngsa_res2['label'] = ngsa_res2['predictor'].map(label_map)
ngsa_res2['ci95'] = 1.96 * ngsa_res2['SE']
ngsa_res2['sig'] = ngsa_res2['p_value'] < 0.05

colors = []
for _, r in ngsa_res2.iterrows():
    if r['predictor'] == 'ko_per_mb_total_z': colors.append(C['null'])
    elif r['predictor'] == 'ngsa_Cu_ppm_z':   colors.append(C['cu'])
    elif r['predictor'] == 'ngsa_Zn_ppm_z':   colors.append(C['zn'])
    elif r['predictor'] == 'ngsa_Pb_ppm_z':   colors.append(C['pb'])
    elif r['predictor'] == 'ngsa_Ni_ppm_z':   colors.append(C['ni'])
    else: colors.append(C['null'])

fig, ax = plt.subplots(figsize=(7, 4.5))
y_pos = np.arange(len(ngsa_res2))[::-1]

for i, (_, row) in enumerate(ngsa_res2.iterrows()):
    facecolor = colors[i] if row['sig'] else 'white'
    edgecolor = colors[i]
    ax.errorbar(row['beta'], y_pos[i], xerr=row['ci95'],
                fmt='none', ecolor=edgecolor, elinewidth=1.8, capsize=5, capthick=1.8)
    ax.plot(row['beta'], y_pos[i], 'o', markersize=10,
            mfc=facecolor, mec=edgecolor, mew=2.0)
    # Significance annotation
    if row['sig']:
        ax.text(row['beta'] + row['ci95'] + 0.0003, y_pos[i],
                f"p={row['p_value']:.3f}*", va='center', fontsize=9, color=edgecolor)
    else:
        ax.text(row['beta'] + row['ci95'] + 0.0003, y_pos[i],
                f"p={row['p_value']:.2f} ns", va='center', fontsize=9, color='#777777')

ax.axvline(0, color='black', linewidth=1.0, linestyle='--', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(ngsa_res2['label'], fontsize=11)
ax.set_xlabel('PGLS β (biome_H_std ~ predictor, n=482 genera)', fontsize=11)
ax.set_title('AusMicrobiome: Effect of metal gene density and\nsoil metal exposure on niche breadth', fontsize=11, pad=8)
ax.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1] + 0.004)

# Section dividers
ax.axhline(y_pos[0] + 0.5, color='#dddddd', linewidth=0.8)

fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig2_ngsa_forest_plot.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig2_ngsa_forest_plot.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 2 saved.')

Fig 2 saved.


## 4  Figure 3 — Genus-level NGSA Cu vs Levins' B

In [5]:
# Genus-level scatter: ngsa_Cu_ppm (raw) vs biome_H_std
ngsa_input = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_levinsB_ngsa_pgls_input.csv')
ngsa_input = ngsa_input.dropna(subset=['biome_H_std', 'ngsa_Cu_ppm'])

# PGLS β=-0.0101 on z-scored Cu, SE=0.0043
cu_std = ngsa_input['ngsa_Cu_ppm'].std()
cu_mean = ngsa_input['ngsa_Cu_ppm'].mean()
beta_cu_raw = -0.0101 / cu_std   # convert z-score β to raw-scale β
intercept_cu = ngsa_input['biome_H_std'].mean() - beta_cu_raw * cu_mean

x_range_cu = np.linspace(ngsa_input['ngsa_Cu_ppm'].quantile(0.01),
                          ngsa_input['ngsa_Cu_ppm'].quantile(0.99), 200)
y_fit_cu = intercept_cu + beta_cu_raw * x_range_cu

# Colour points by phylum if available; otherwise single color
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(ngsa_input['ngsa_Cu_ppm'], ngsa_input['biome_H_std'],
           alpha=0.4, s=18, color=C['aus'], zorder=2, linewidths=0)
ax.plot(x_range_cu, y_fit_cu, color='#d73027', linewidth=2.2, zorder=3,
        label=f'PGLS: β = −0.0101 (SE 0.0043), p = 0.019')
ax.axhline(0, color='#cccccc', linewidth=0.6, linestyle='--')

ax.set_xlabel('NGSA Copper (ppm, soil at nearest sampling site)', fontsize=11)
ax.set_ylabel('Niche breadth, Levins\' B (standardized)', fontsize=11)
ax.set_title(
    f'AusMicrobiome: NGSA Copper vs genus niche breadth\n'
    f'n = {len(ngsa_input)} genera (≥1 AusMicrobiome detection)',
    fontsize=11, pad=8)
ax.legend(fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig3_ngsa_cu_scatter.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig3_ngsa_cu_scatter.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 3 saved.')

Fig 3 saved.


## 5  Figure 4 — Moran's I by biome (spatial autocorrelation)

In [6]:
moran = pd.read_csv(f'{BASE}/data/moran_i_metal_genes.csv')
moran['sig'] = moran['p_perm'] < 0.05
# Pivot: biome × variable
pivot = moran.pivot(index='biome', columns='variable', values='morans_I')
piv_p = moran.pivot(index='biome', columns='variable', values='p_perm')

biomes = ['Soil', 'Marine', 'Marine Sediment']
variables = ['metal_type_diversity', 'total_metal_genes']
var_labels = ['Metal type diversity', 'Total metal genes']
biome_colors = [C['soil'], C['marine'], C['sediment']]

x = np.arange(len(variables))
width = 0.22
offsets = [-width, 0, width]

fig, ax = plt.subplots(figsize=(7, 4.5))
for j, (biome, offset, color) in enumerate(zip(biomes, offsets, biome_colors)):
    vals = [pivot.loc[biome, v] for v in variables]
    p_vals = [piv_p.loc[biome, v] for v in variables]
    bars = ax.bar(x + offset, vals, width, label=biome, color=color, alpha=0.85)
    for bar, p in zip(bars, p_vals):
        sig_str = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                sig_str, ha='center', va='bottom', fontsize=9,
                color='#333333' if p < 0.05 else '#999999')

ax.set_xticks(x)
ax.set_xticklabels(var_labels, fontsize=11)
ax.set_ylabel("Moran's I (spatial autocorrelation)", fontsize=11)
ax.set_title("Spatial autocorrelation of metal resistance genes\nin MGnify environmental MAGs", fontsize=11, pad=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.legend(fontsize=10, frameon=False, title='Biome')

# Expected I reference
ax.axhline(0.0, color='#cccccc', linewidth=0.6, linestyle='--')

fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig4_moran_i_biome.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig4_moran_i_biome.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 4 saved.')

Fig 4 saved.


## 6  Figure 5 — Mining proximity vs metal gene diversity (MGnify soil MAGs)

In [7]:
mags = pd.read_csv(f'{BASE}/data/mags_annotated_geo.csv',
                   usecols=['biome_name', 'lat', 'lon', 'n_metal_types', 'dist_mine_km'])
soil = mags[mags['biome_name'].str.contains('soil|Soil|terrestrial|Terrestrial',
                                              na=False, case=False)].copy()
soil = soil[soil['dist_mine_km'].notna() & soil['n_metal_types'].notna()].copy()

print(f'Soil MAGs with mine proximity data: {len(soil):,}')
print(f'dist_mine_km range: {soil.dist_mine_km.min():.1f} – {soil.dist_mine_km.max():.1f} km')

# Log-scale distance, bin into deciles
soil['log_dist'] = np.log10(soil['dist_mine_km'].clip(lower=0.1))
n_bins = 12
soil['dist_bin'] = pd.qcut(soil['dist_mine_km'], q=n_bins, labels=False, duplicates='drop')

binned = soil.groupby('dist_bin').agg(
    mean_dist    = ('dist_mine_km', 'median'),
    mean_n_types = ('n_metal_types', 'mean'),
    se_n_types   = ('n_metal_types', lambda x: x.std() / np.sqrt(len(x))),
    n            = ('n_metal_types', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(6.5, 5.0))
ax.errorbar(np.log10(binned['mean_dist']), binned['mean_n_types'],
            yerr=1.96 * binned['se_n_types'],
            fmt='o-', color=C['soil'], markersize=7, linewidth=1.8,
            elinewidth=1.2, capsize=4, capthick=1.2, zorder=3)

# Trend line (OLS on log dist)
from numpy.polynomial.polynomial import polyfit as nppolyfit
log_dist_all = np.log10(soil['dist_mine_km'].clip(lower=0.1))
coef = np.polyfit(log_dist_all, soil['n_metal_types'], 1)
x_trend = np.linspace(log_dist_all.min(), log_dist_all.max(), 200)
ax.plot(x_trend, np.polyval(coef, x_trend), '--', color='#d73027',
        linewidth=1.5, alpha=0.7, label=f'OLS trend (slope={coef[0]:.3f}/log-km)')

xticks = [1, 10, 50, 100, 500, 2000, 8000]
ax.set_xticks(np.log10(xticks))
ax.set_xticklabels(xticks)
ax.set_xlabel('Distance to nearest mine (km, log scale)', fontsize=11)
ax.set_ylabel('Mean metal type diversity (n types)', fontsize=11)
ax.set_title('MGnify soil MAGs: mining proximity vs\nmetal resistance gene diversity', fontsize=11, pad=8)
ax.legend(fontsize=9, frameon=False)

# Spearman annotation
from scipy.stats import spearmanr
rho, pval = spearmanr(soil['dist_mine_km'], soil['n_metal_types'])
ax.text(0.97, 0.97, f'Spearman ρ = {rho:.3f}\np = {pval:.1e}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig5_mine_proximity_scatter.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig5_mine_proximity_scatter.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 5 saved.')

Soil MAGs with mine proximity data: 7,939
dist_mine_km range: 1.5 – 3873.3 km


Fig 5 saved.


## 7  NGSA distance threshold sensitivity

Re-aggregate genus-level NGSA Cu from sample-level data at different max_dist_km thresholds.
Requires loading the 16S OTU table to get genus × sample detection matrix.

In [8]:
print('Loading AusMicrobiome OTU table...')
otu = pd.read_csv(f'{BASE}/data/aus_microbiome/BASE_16S_OTU.csv.gz',
                  index_col=0, compression='gzip')
otu.index.name = 'OTUId'
print(f'OTU table: {otu.shape[0]} OTUs × {otu.shape[1]} samples')

print('Loading taxonomy...')
tax = pd.read_excel(f'{BASE}/data/aus_microbiome/BASE_16S_taxonomy.xlsx',
                    usecols=['OTUId', 'genus'])
tax['genus_clean'] = (
    tax['genus']
    .str.replace('g__', '', regex=False)
    .str.strip()
    .str.lower()
    .replace({'unclassified': np.nan, '': np.nan})
)
tax = tax.dropna(subset=['genus_clean'])
print(f'Taxonomy: {len(tax)} OTUs with genus assignment')

Loading AusMicrobiome OTU table...


OTU table: 91929 OTUs × 1023 samples
Loading taxonomy...


Taxonomy: 18152 OTUs with genus assignment


In [9]:
# Build genus × sample presence matrix
# Merge OTU table with genus assignments
otu_w_genus = otu.join(tax.set_index('OTUId')['genus_clean'], how='inner')
print(f'OTUs with genus: {len(otu_w_genus)}')

# Sum counts by genus → genus × sample matrix
genus_counts = otu_w_genus.groupby('genus_clean').sum()
# Presence/absence (any detection)
genus_presence = (genus_counts > 0)
print(f'Genus presence matrix: {genus_presence.shape[0]} genera × {genus_presence.shape[1]} samples')

# Map OTU sample IDs (numeric) to NGSA data
aus_ngsa = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_sample_ngsa.csv')
aus_ngsa['sample_num'] = aus_ngsa['Sample_ID'].str.replace('102.100.100/', '', regex=False)

# Build lookups: sample_num → NGSA Cu_ppm and dist_km
cu_lookup   = dict(zip(aus_ngsa['sample_num'], aus_ngsa['ngsa_Cu_ppm']))
dist_lookup = dict(zip(aus_ngsa['sample_num'], aus_ngsa['ngsa_dist_km']))
zn_lookup   = dict(zip(aus_ngsa['sample_num'], aus_ngsa['ngsa_Zn_ppm']))
pb_lookup   = dict(zip(aus_ngsa['sample_num'], aus_ngsa['ngsa_Pb_ppm']))
ni_lookup   = dict(zip(aus_ngsa['sample_num'], aus_ngsa['ngsa_Ni_ppm']))

print(f'NGSA lookup: {len(cu_lookup)} samples')
sample_cols = [str(c) for c in genus_presence.columns]
n_matched = sum(1 for c in sample_cols if c in cu_lookup)
print(f'OTU samples with NGSA match: {n_matched}/{len(sample_cols)}')

OTUs with genus: 18152


Genus presence matrix: 933 genera × 1023 samples
NGSA lookup: 1663 samples
OTU samples with NGSA match: 1019/1023


In [10]:
def aggregate_ngsa_at_threshold(genus_presence, metal_lookup, dist_lookup, threshold_km,
                                 sample_cols):
    """Mean metal ppm per genus across detected samples within threshold."""
    metal_vals = np.array([
        metal_lookup.get(c, np.nan)
        if dist_lookup.get(c, np.inf) <= threshold_km
        else np.nan
        for c in sample_cols
    ], dtype=float)

    presence = genus_presence.values
    valid = ~np.isnan(metal_vals)
    mask = presence & valid[np.newaxis, :]
    genus_sum   = np.nansum(np.where(mask, metal_vals[np.newaxis, :], np.nan), axis=1)
    genus_count = mask.sum(axis=1)
    genus_mean  = np.where(genus_count > 0, genus_sum / genus_count, np.nan)
    return pd.Series(genus_mean, index=genus_presence.index)


def run_pgls_on_df(df_input, tree_path, pgls_script, cwd):
    """Write temp CSV, run R PGLS script, return results DataFrame."""
    with tempfile.TemporaryDirectory() as tmp:
        inp = os.path.join(tmp, 'pgls_input.csv')
        out = os.path.join(tmp, 'pgls_output.csv')
        df_input.to_csv(inp, index=False)
        r = subprocess.run(
            [RSCRIPT, pgls_script, inp, tree_path, out],
            capture_output=True, text=True, cwd=cwd
        )
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-2000:])
        return pd.read_csv(out)


response_df = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_ngsa_pgls_focused.csv',
                           usecols=['genus_lower', 'biome_H_std'])

thresholds = [30, 50, 75, 100, 150, 200]
metals = {'Cu': (cu_lookup, 'ngsa_Cu_ppm'), 'Zn': (zn_lookup, 'ngsa_Zn_ppm'),
          'Pb': (pb_lookup, 'ngsa_Pb_ppm'), 'Ni': (ni_lookup, 'ngsa_Ni_ppm')}

sens_rows = []
for threshold in thresholds:
    print(f'  Threshold {threshold} km ...', end='')
    row_d = {'threshold_km': threshold}

    metal_genus = {}
    for metal_name, (lookup, col_name) in metals.items():
        genus_mean = aggregate_ngsa_at_threshold(
            genus_presence, lookup, dist_lookup, threshold, sample_cols)
        metal_genus[metal_name] = genus_mean.rename(col_name)

    metal_df = pd.concat(metal_genus.values(), axis=1)
    metal_df.index.name = 'genus_lower'
    metal_df = metal_df.reset_index()

    combined = response_df.merge(metal_df, on='genus_lower', how='inner')
    combined = combined.dropna(subset=['biome_H_std'])
    n_with_ngsa = combined.dropna(subset=['ngsa_Cu_ppm']).shape[0]

    for col in ['ngsa_Cu_ppm', 'ngsa_Zn_ppm', 'ngsa_Pb_ppm', 'ngsa_Ni_ppm']:
        m, s = combined[col].mean(), combined[col].std()
        combined[col + '_z'] = (combined[col] - m) / s if s > 0 else 0.0

    pgls_df = combined[['genus_lower', 'biome_H_std',
                          'ngsa_Cu_ppm_z', 'ngsa_Zn_ppm_z',
                          'ngsa_Pb_ppm_z', 'ngsa_Ni_ppm_z']].dropna()
    n_genera = len(pgls_df)
    print(f' n_genera={n_genera}', end='')

    if n_genera < 50:
        print(' → too few, skip')
        row_d.update({'n_genera': n_genera, 'Cu_beta': np.nan, 'Cu_p': np.nan})
        sens_rows.append(row_d)
        continue

    try:
        res = run_pgls_on_df(pgls_df, TREE, PGLS_R, BASE)
        for metal in ['Cu', 'Zn', 'Pb', 'Ni']:
            col_z = f'ngsa_{metal}_ppm_z'
            m_row = res[res['predictor'] == col_z]
            if len(m_row):
                row_d[f'{metal}_beta'] = m_row['beta'].values[0]
                row_d[f'{metal}_SE']   = m_row['SE'].values[0]
                row_d[f'{metal}_p']    = m_row['p_value'].values[0]
        row_d['n_genera'] = n_genera
        print(f' Cu_β={row_d.get("Cu_beta", np.nan):.4f} p={row_d.get("Cu_p", np.nan):.3f}')
    except Exception as e:
        print(f' PGLS failed: {str(e)[:100]}')
        row_d.update({'n_genera': n_genera, 'Cu_beta': np.nan, 'Cu_p': np.nan})

    sens_rows.append(row_d)

sens_df = pd.DataFrame(sens_rows)
print()
print('Sensitivity results:')
print(sens_df[['threshold_km','n_genera','Cu_beta','Cu_p','Zn_beta','Zn_p','Pb_beta','Pb_p']].to_string(
    index=False, float_format='%.4f'))
sens_df.to_csv(f'{BASE}/data/ngsa_threshold_sensitivity.csv', index=False)
print('\nSaved: data/ngsa_threshold_sensitivity.csv')

  Threshold 30 km ... n_genera=462

 Cu_β=-0.0089 p=0.046
  Threshold 50 km ... n_genera=480

 Cu_β=-0.0112 p=0.010
  Threshold 75 km ... n_genera=480

 Cu_β=-0.0092 p=0.034
  Threshold 100 km ... n_genera=481

 Cu_β=-0.0063 p=0.154
  Threshold 150 km ... n_genera=482

 Cu_β=-0.0101 p=0.020
  Threshold 200 km ... n_genera=482

 Cu_β=-0.0101 p=0.019

Sensitivity results:
 threshold_km  n_genera  Cu_beta   Cu_p  Zn_beta   Zn_p  Pb_beta   Pb_p
           30       462  -0.0089 0.0463  -0.0055 0.2394  -0.0064 0.1547
           50       480  -0.0112 0.0103  -0.0105 0.0162  -0.0015 0.7262
           75       480  -0.0092 0.0340  -0.0086 0.0479  -0.0015 0.7284
          100       481  -0.0063 0.1541  -0.0074 0.0870  -0.0012 0.7877
          150       482  -0.0101 0.0196  -0.0096 0.0284  -0.0089 0.0398
          200       482  -0.0101 0.0194  -0.0097 0.0268  -0.0089 0.0395

Saved: data/ngsa_threshold_sensitivity.csv


## 8  Figure 6 — NGSA distance threshold sensitivity

In [11]:
sens = pd.read_csv(f'{BASE}/data/ngsa_threshold_sensitivity.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: β ± 95% CI per metal per threshold
metal_plot = [
    ('Cu', C['cu'], 'Copper'),
    ('Zn', C['zn'], 'Zinc'),
    ('Pb', C['pb'], 'Lead'),
    ('Ni', C['ni'], 'Nickel'),
]
x = sens['threshold_km'].values

for metal, color, label in metal_plot:
    beta_col = f'{metal}_beta'
    se_col   = f'{metal}_SE'
    if beta_col not in sens.columns: continue
    beta = sens[beta_col].values
    se   = sens[se_col].values if se_col in sens.columns else np.zeros_like(beta)
    ax1.plot(x, beta, 'o-', color=color, label=label, linewidth=1.8, markersize=6)
    ax1.fill_between(x, beta - 1.96*se, beta + 1.96*se, alpha=0.12, color=color)

ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_xlabel('NGSA max distance threshold (km)', fontsize=11)
ax1.set_ylabel('PGLS β', fontsize=11)
ax1.set_title('β coefficient stability\nacross distance thresholds', fontsize=11, pad=8)
ax1.legend(fontsize=9, frameon=False)

# Right: p-values (log scale)
for metal, color, label in metal_plot:
    p_col = f'{metal}_p'
    if p_col not in sens.columns: continue
    p_vals = sens[p_col].values
    valid = ~np.isnan(p_vals) & (p_vals > 0)
    ax2.plot(x[valid], -np.log10(p_vals[valid]), 'o-', color=color,
             label=label, linewidth=1.8, markersize=6)

ax2.axhline(-np.log10(0.05), color='#d73027', linewidth=1.2,
            linestyle='--', label='p = 0.05')
ax2.set_xlabel('NGSA max distance threshold (km)', fontsize=11)
ax2.set_ylabel('−log₁₀(p)', fontsize=11)
ax2.set_title('Significance stability\nacross distance thresholds', fontsize=11, pad=8)
ax2.legend(fontsize=9, frameon=False)

fig.suptitle('Sensitivity of NGSA PGLS to distance threshold', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig6_ngsa_threshold_sensitivity.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig6_ngsa_threshold_sensitivity.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 6 saved.')

Fig 6 saved.


## 9  Figure 7 — Genus detection frequency threshold sensitivity

In [12]:
BASELINE_THRESHOLD = 200
print(f'Detection frequency sensitivity (NGSA threshold={BASELINE_THRESHOLD} km)')

cu_vals = np.array([
    cu_lookup.get(c, np.nan)
    if dist_lookup.get(c, np.inf) <= BASELINE_THRESHOLD
    else np.nan
    for c in sample_cols
], dtype=float)
valid_mask = ~np.isnan(cu_vals)
presence   = genus_presence.values
mask       = presence & valid_mask[np.newaxis, :]

n_ngsa_detections = mask.sum(axis=1)
genus_cu_sum      = np.nansum(np.where(mask, cu_vals[np.newaxis, :], np.nan), axis=1)
genus_cu_mean     = np.where(n_ngsa_detections > 0, genus_cu_sum / n_ngsa_detections, np.nan)

detection_df = pd.DataFrame({
    'genus_lower': genus_presence.index,
    'ngsa_Cu_ppm': genus_cu_mean,
    'n_ngsa_detections': n_ngsa_detections
})

min_det_rows = []
for min_det in [1, 2, 5, 10, 20, 30]:
    sub = detection_df[detection_df['n_ngsa_detections'] >= min_det].copy()
    sub = sub.merge(response_df, on='genus_lower', how='inner').dropna()
    m_cu, s_cu = sub['ngsa_Cu_ppm'].mean(), sub['ngsa_Cu_ppm'].std()
    sub['ngsa_Cu_ppm_z'] = (sub['ngsa_Cu_ppm'] - m_cu) / s_cu
    pgls_in = sub[['genus_lower', 'biome_H_std', 'ngsa_Cu_ppm_z']].dropna()
    n = len(pgls_in)
    print(f'  min_det={min_det:2d}: n_genera={n}', end='')
    if n < 50:
        print(' → skip')
        min_det_rows.append({'min_det': min_det, 'n_genera': n, 'Cu_beta': np.nan, 'Cu_p': np.nan})
        continue
    try:
        res = run_pgls_on_df(pgls_in, TREE, PGLS_R, BASE)
        r = res[res['predictor'] == 'ngsa_Cu_ppm_z'].iloc[0]
        print(f' β={r["beta"]:.4f} p={r["p_value"]:.3f}')
        min_det_rows.append({'min_det': min_det, 'n_genera': n,
                              'Cu_beta': r['beta'], 'Cu_SE': r['SE'], 'Cu_p': r['p_value']})
    except Exception as e:
        print(f' failed: {str(e)[:80]}')
        min_det_rows.append({'min_det': min_det, 'n_genera': n, 'Cu_beta': np.nan, 'Cu_p': np.nan})

min_det_df = pd.DataFrame(min_det_rows)
print()
print(min_det_df.to_string(index=False, float_format='%.4f'))

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4))
valid = min_det_df.dropna(subset=['Cu_beta'])
ax_a.plot(valid['min_det'], valid['Cu_beta'], 'o-', color=C['cu'], linewidth=2, markersize=8)
if 'Cu_SE' in valid.columns and not valid['Cu_SE'].isna().all():
    ax_a.fill_between(valid['min_det'],
                       valid['Cu_beta'] - 1.96*valid['Cu_SE'],
                       valid['Cu_beta'] + 1.96*valid['Cu_SE'], alpha=0.15, color=C['cu'])
ax_a.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax_a.set_xlabel('Min NGSA-matched detections per genus', fontsize=11)
ax_a.set_ylabel('PGLS β (Cu)', fontsize=11)
ax_a.set_title('β stability across\ndetection frequency thresholds', fontsize=11)

if not valid.empty and not valid['Cu_p'].isna().all():
    ax_b.plot(valid['min_det'], -np.log10(valid['Cu_p'].clip(1e-10)), 'o-',
              color=C['cu'], linewidth=2, markersize=8)
ax_b.axhline(-np.log10(0.05), color='#d73027', linewidth=1.2, linestyle='--', label='p=0.05')
ax_b.set_xlabel('Min NGSA-matched detections per genus', fontsize=11)
ax_b.set_ylabel('−log₁₀(p)', fontsize=11)
ax_b.set_title('Significance stability', fontsize=11)
ax_b.legend(fontsize=9, frameon=False)
ax_b2 = ax_b.twinx()
ax_b2.plot(valid['min_det'], valid['n_genera'], 's--', color='#888888',
           linewidth=1, markersize=6, alpha=0.6, label='n genera')
ax_b2.set_ylabel('n genera', fontsize=9, color='#888888')
ax_b2.tick_params(axis='y', labelcolor='#888888')
ax_b2.legend(fontsize=9, frameon=False, loc='lower right')

fig.suptitle('Sensitivity to minimum detection frequency (NGSA Cu)', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig7_detection_freq_sensitivity.pdf', bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig7_detection_freq_sensitivity.png', bbox_inches='tight', dpi=200)
plt.show()
print('Fig 7 saved.')

Detection frequency sensitivity (NGSA threshold=200 km)
  min_det= 1: n_genera=482

 β=-0.0101 p=0.019
  min_det= 2: n_genera=481

 β=-0.0105 p=0.015
  min_det= 5: n_genera=451

 β=-0.0138 p=0.002
  min_det=10: n_genera=386

 β=-0.0134 p=0.002
  min_det=20: n_genera=333

 β=-0.0136 p=0.007
  min_det=30: n_genera=299

 β=-0.0107 p=0.047

 min_det  n_genera  Cu_beta  Cu_SE   Cu_p
       1       482  -0.0101 0.0043 0.0194
       2       481  -0.0105 0.0043 0.0150
       5       451  -0.0138 0.0045 0.0024
      10       386  -0.0134 0.0043 0.0019
      20       333  -0.0136 0.0050 0.0073
      30       299  -0.0107 0.0054 0.0474


Fig 7 saved.


## 10  Summary

In [13]:
# FDR summary
print('=== FDR Correction Summary ===\n')
ngsa_res_fresh = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_ngsa_pgls_results.csv')
ngsa_res_fresh = ngsa_res_fresh.sort_values('p_value').reset_index(drop=True)
m = len(ngsa_res_fresh)
ngsa_res_fresh['rank'] = np.arange(1, m+1)
ngsa_res_fresh['q_BH'] = (ngsa_res_fresh['p_value'] * m / ngsa_res_fresh['rank']).clip(upper=1.0)
ngsa_res_fresh['q_BH'] = ngsa_res_fresh['q_BH'][::-1].cummin()[::-1]
ngsa_res_fresh['label'] = ngsa_res_fresh['predictor'].map(label_map)
ngsa_res_fresh['fdr10'] = ngsa_res_fresh['q_BH'] < 0.10

print(ngsa_res_fresh[['label', 'beta', 'SE', 'p_value', 'q_BH', 'fdr10']].to_string(
    index=False, float_format='%.4f'))

print('\n=== Sensitivity Analysis Summary ===\n')
if os.path.exists(f'{BASE}/data/ngsa_threshold_sensitivity.csv'):
    st = pd.read_csv(f'{BASE}/data/ngsa_threshold_sensitivity.csv')
    print('Distance threshold sensitivity (Cu β and p):')
    print(st[['threshold_km', 'n_genera', 'Cu_beta', 'Cu_p']].to_string(index=False, float_format='%.4f'))

print('\n=== Figures produced ===\n')
for f in sorted(os.listdir(FIGDIR)):
    if f.endswith('.png'):
        size = os.path.getsize(f'{FIGDIR}/{f}')
        print(f'  {f} ({size/1024:.0f} KB)')

=== FDR Correction Summary ===

           label    beta     SE  p_value   q_BH  fdr10
     NGSA Copper -0.0101 0.0043   0.0194 0.0691   True
       NGSA Zinc -0.0096 0.0043   0.0268 0.0691   True
       NGSA Lead -0.0089 0.0043   0.0395 0.0691   True
     NGSA Nickel -0.0087 0.0044   0.0461 0.0691   True
     NGSA Cobalt  0.0019 0.0043   0.6581 0.6674  False
Metal KO density -0.0023 0.0054   0.6674 0.6674  False

=== Sensitivity Analysis Summary ===

Distance threshold sensitivity (Cu β and p):
 threshold_km  n_genera  Cu_beta   Cu_p
           30       462  -0.0089 0.0463
           50       480  -0.0112 0.0103
           75       480  -0.0092 0.0340
          100       481  -0.0063 0.1541
          150       482  -0.0101 0.0196
          200       482  -0.0101 0.0194

=== Figures produced ===

  fig1_primary_pgls_scatter.png (142 KB)
  fig2_ngsa_forest_plot.png (99 KB)
  fig3_ngsa_cu_scatter.png (153 KB)
  fig4_moran_i_biome.png (74 KB)
  fig5_mine_proximity_scatter.png (116 KB)
  f